# LISS-IV Cloud Removal — Collaborative Training with DagsHub & MLflow

This notebook allows team members to train **U-Net** and **Pix2Pix Conditional GAN (cGAN)** models on free Google Colab GPUs (T4), streaming all parameters, losses, metrics, and checkpoints to a **centralized, shared DagsHub MLflow dashboard** (`https://dagshub.com/Mnthn2326/liss4-cloud-removal`).

## Step 1: GPU Hardware Check

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"[OK] GPU Detected: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
else:
    print("[WARNING] GPU not detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

## Step 2: Clone Repository and Install Dependencies

In [ ]:
import os

if not os.path.exists("liss4-cloud-removal"):
    !git clone https://github.com/Mnthn2326/liss4-cloud-removal.git

%cd liss4-cloud-removal

!pip install -q -e .
!pip install -q dagshub mlflow
print("[OK] Environment setup complete.")

## Step 3: Connect to Shared DagsHub MLflow Tracking
Each teammate enters their name so the runs in DagsHub are tagged with the contributor who trained them.

In [ ]:
import os
import dagshub

# >>> CHANGE THIS TO YOUR NAME / TEAMMATE NAME <<<
TEAMMATE_NAME = "Manthan"  # e.g., "Manthan", "Alice", "Bob"
os.environ["MLFLOW_USER"] = TEAMMATE_NAME

# Initialize DagsHub MLflow tracking
dagshub.init(repo_owner="Mnthn2326", repo_name="liss4-cloud-removal", mlflow=True)

print(f"[OK] Connected to DagsHub MLflow! Logging runs under user: {TEAMMATE_NAME}")
print(f"Tracking URI: {os.environ.get('MLFLOW_TRACKING_URI')}")

## Step 4: Acquire Dataset
Verifies and downloads the SEN12MS-CR paired optical training subset if not present.

In [ ]:
from pathlib import Path

manifest_path = Path("data/raw/sen12mscr/manifest.json")
if not manifest_path.is_file():
    print("Downloading SEN12MS-CR paired optical dataset...")
    !python -m cloud_removal.acquire_sen12mscr
else:
    print("[OK] Dataset manifest already verified on disk.")

---
# Step 5: Choose Which Model to Train

Your team can run any of the following training tracks on T4 GPU:

### Track A: Train Conventional U-Net Baseline

In [ ]:
# Train Standard RGB U-Net Baseline
!python -m cloud_removal.train --config configs/train_unet.yaml

In [ ]:
# OR Train False-Color Composite (FCC) U-Net (LISS-IV sensor aligned: NIR, Red, Green)
!python -m cloud_removal.train --config configs/train_fcc_unet.yaml

### Track B: Train Pix2Pix Conditional GAN (cGAN)
Trains U-Net Generator + PatchGAN Discriminator with Adversarial + L1 Content Loss.

In [ ]:
# Train Standard RGB Pix2Pix cGAN
!python -m cloud_removal.train_cgan --config configs/train_cgan.yaml

In [ ]:
# OR Train False-Color Composite (FCC) Pix2Pix cGAN (LISS-IV sensor aligned: NIR, Red, Green)
!python -m cloud_removal.train_cgan --config configs/train_fcc_cgan.yaml

---
## Step 6: Visual Inference & Comparison
Generate side-by-side comparison figures (Cloudy Input vs. Removed Prediction vs. Ground Truth Clear Reference) from the best checkpoint.

In [ ]:
from IPython.display import Image, display

# Select the checkpoint to test (e.g. cgan or unet)
checkpoint_to_test = "checkpoints/cgan/best_cgan_generator.pt"
output_fig = "reports/figures/colab_inference_result.png"

!python -m cloud_removal.infer \
    --checkpoint $checkpoint_to_test \
    --split val \
    --index 0 \
    --output $output_fig

display(Image(output_fig))

## Step 7: Download Checkpoint to Local Machine
Save your trained model checkpoint to your local machine for deployment in Docker / FastAPI.

In [ ]:
from google.colab import files
import os

if os.path.exists(checkpoint_to_test):
    print(f"Downloading {checkpoint_to_test}...")
    files.download(checkpoint_to_test)
else:
    print("Checkpoint file not found.")